In [7]:

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [8]:

#Preprocessing the data
data= pd.read_csv('Churn_Modelling.csv')
data= data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender= LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoder,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))


data= pd.concat([data.drop(['Geography'], axis=1), geo_encoded_df], axis=1)

X=data.drop(['Exited'], axis=1)
y=data['Exited']

X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2, random_state=42)
scaler= StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Save the encoders  and scalers
with open ('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open ('onehot_encoder_geo.pkl', 'wb') as file:    
    pickle.dump(onehot_encoder_geo, file)

with open ('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)    

In [9]:
#Define a function to create the model and try different parameters

def create_model(neurons=32, layers=1):
    model= Sequential()
    model.add(Dense(neurons, input_shape=(X_train.shape[1], ),  activation='relu'))
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    return model

In [10]:
#Create a KerasClassifier for use in GridSearchCV
model= KerasClassifier(model= create_model, layers=1, neurons=32, verbose=1)



In [11]:
#Define the frid search parameters
param_grid  = {
    'neurons' : [16, 32, 64, 128],
    'layers' : [1, 2, 3],
    'batch_size' :[10, 20],
    'epochs' : [50, 100]
    }

In [12]:
#Perform grid search 
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result= grid.fit(X_train, y_train)

#Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 48 candidates, totalling 144 fits


d:\Swati\AIBasics\annclassification\myenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8002 - loss: 0.4592
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8379 - loss: 0.3952
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8460 - loss: 0.3712
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8521 - loss: 0.3577
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8566 - loss: 0.3513
Epoch 6/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8570 - loss: 0.3470
Epoch 7/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8600 - loss: 0.3437
Epoch 8/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8605 - loss: 0.3436
Epoch 9/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8586 - loss: 0.3418
Epoch 10/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8605 - loss: 0.3408
Epoch 11/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8610 - loss: 0.3406
Epoch 12/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step